# Case study: helpful-votes regression and Telco churn

Session 6 · Introduction to machine learning with regression. Author: course team, licence CC-BY-4.0.

The notebook follows the three practice tasks of the session:

1. **Block 1**: map the review-sentiment task to the ten lifecycle steps; split the data and model the number of helpful votes with linear regression; evaluate with MAE, RMSE and R².
2. **Block 2**: training versus test error for increasing model complexity; least squares versus robust regression (Huber, quantile) on helpful votes.
3. **Block 3**: predict churn probability for the IBM Telco customers with logistic regression and interpret the coefficients.

Cells marked **Exercise** are for you. Theory: `../theory/01-lifecycle-and-linear-regression.md` to `03-logistic-regression.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import HuberRegressor, LinearRegression, LogisticRegression
from sklearn.metrics import (confusion_matrix, mean_absolute_error, r2_score,
                             root_mean_squared_error)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
reviews = pd.read_parquet(ROOT / "case-study" / "data" / "train_sample.parquet")
reviews["n_words"] = reviews["text"].str.split().str.len()
reviews["log_words"] = np.log1p(reviews["n_words"])
reviews["log_votes"] = np.log1p(reviews["helpful_vote"])
reviews["n_title_words"] = reviews["title"].str.split().str.len()
reviews["age_years"] = (pd.Timestamp("2022-01-01") - reviews["date"]).dt.days / 365.25
reviews["verified"] = reviews["verified_purchase"].astype(int)
print(reviews.shape)

## Block 1 · Lifecycle, split and linear regression

### 1.1 The review-sentiment task in ten steps

**Exercise.** Complete the table for the leaderboard task (predict neg/neu/pos from text and metadata; metric macro-F1).

| Step | What it means for the sentiment task | Session |
|---|---|---|
| 01 Problem definition | target = label (neg/neu/pos); metric = macro-F1; baseline = always "pos" (0.26) | S6 |
| 02 Data collection | … | S2, S3 |
| 03 Data cleaning and preprocessing | … | S4 |
| 04 Exploratory data analysis | … | S5 |
| 05 Feature engineering and selection | … | |
| 06 Model selection | … | |
| 07 Model training | … | |
| 08 Model evaluation and tuning | … | |
| 09 Model deployment | … | |
| 10 Model monitoring and maintenance | … | |

### 1.2 Train–test split

Target: `log_votes` = log(1 + helpful votes). The raw count is extremely skewed (mean 1.3, maximum 7,326), so least squares on the raw scale would be dominated by a handful of reviews.

In [ ]:
features = ["log_words", "n_images", "verified", "rating", "n_title_words", "age_years"]
train, test = train_test_split(reviews, test_size=0.2, random_state=42)
print(len(train), len(test))
train[features + ["log_votes"]].describe().round(2)

### 1.3 Simple and multiple linear regression

In [ ]:
fit_simple = smf.ols("log_votes ~ log_words", data=train).fit()
fit_multi = smf.ols("log_votes ~ " + " + ".join(features), data=train).fit()
print(fit_simple.params.round(4))
fit_multi.summary().tables[1]

**Exercise.** Interpret the coefficients of `log_words`, `verified` and `age_years` in one sentence each ("holding the other features fixed, …"). Why might `age_years` be positive?

### 1.4 Metrics on the test set, compared with the baseline

In [ ]:
def evaluate(name, model, cols, target="log_votes"):
    model.fit(train[cols], train[target])
    pred_train, pred_test = model.predict(train[cols]), model.predict(test[cols])
    return {"model": name,
            "MAE": mean_absolute_error(test[target], pred_test),
            "RMSE": root_mean_squared_error(test[target], pred_test),
            "R2 test": r2_score(test[target], pred_test),
            "R2 train": r2_score(train[target], pred_train)}

results = pd.DataFrame([
    evaluate("baseline (mean)", DummyRegressor(), ["log_words"]),
    evaluate("simple: length", LinearRegression(), ["log_words"]),
    evaluate("multiple: 6 features", LinearRegression(), features),
]).set_index("model").round(3)
results

In [ ]:
model = LinearRegression().fit(train[features], train["log_votes"])
fitted = model.predict(test[features])
resid = test["log_votes"] - fitted
fig, ax = plt.subplots(figsize=(6.5, 4), layout="constrained")
ax.scatter(fitted, resid, s=4, alpha=0.3, color="0.4")
ax.axhline(0, color="black", lw=1)
ax.set(xlabel="fitted log(1 + votes)", ylabel="residual", title="Residuals on the test set");

**Exercise.** Describe the residual plot. Which pattern comes from reviews with zero votes? What does the long upper tail mean?

## Block 2 · Model complexity and robust regression

### 2.1 Training versus test error for increasing complexity

First on toy data (20 points from a cosine curve), then on the reviews.

In [ ]:
rng = np.random.default_rng(0)
x_tr = np.sort(rng.uniform(0, 1, 20)).reshape(-1, 1)
y_tr = np.cos(1.5 * np.pi * x_tr.ravel()) + rng.normal(0, 0.15, 20)
x_te = rng.uniform(0, 1, 1000).reshape(-1, 1)
y_te = np.cos(1.5 * np.pi * x_te.ravel()) + rng.normal(0, 0.15, 1000)

rows = []
for degree in range(1, 16):
    m = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x_tr, y_tr)
    rows.append({"degree": degree,
                 "train RMSE": root_mean_squared_error(y_tr, m.predict(x_tr)),
                 "test RMSE": root_mean_squared_error(y_te, m.predict(x_te))})
curve = pd.DataFrame(rows).set_index("degree")
ax = curve.plot(logy=True, marker="o", figsize=(6.5, 3.8), title="Toy data: training vs test error")
ax.set_ylabel("RMSE (log scale)")
curve.round(3).T

In [ ]:
rows = []
for degree in [1, 2, 3]:
    m = make_pipeline(StandardScaler(), PolynomialFeatures(degree), LinearRegression())
    m.fit(train[features], train["log_votes"])
    rows.append({"degree": degree, "n model columns": m[1].n_output_features_,
                 "train RMSE": root_mean_squared_error(train["log_votes"], m.predict(train[features])),
                 "test RMSE": root_mean_squared_error(test["log_votes"], m.predict(test[features]))})
pd.DataFrame(rows).set_index("degree").round(4)

**Exercise.** Repeat the review experiment with only 500 training reviews (`train.sample(500, random_state=0)`). At which degree does the test error start to rise? What does this say about flexibility relative to the amount of data?

### 2.2 Least squares versus robust regression on helpful votes

On the raw vote count, a few reviews with thousands of votes dominate least squares.

In [ ]:
rows = []
for name, fit in {
    "OLS (mean)": smf.ols("helpful_vote ~ log_words", train).fit(),
    "median (q=0.5)": smf.quantreg("helpful_vote ~ log_words", train).fit(q=0.5),
    "q=0.9": smf.quantreg("helpful_vote ~ log_words", train).fit(q=0.9),
}.items():
    pred = fit.predict(test)
    rows.append({"model": name, "slope": fit.params["log_words"],
                 "MAE": mean_absolute_error(test["helpful_vote"], pred),
                 "RMSE": root_mean_squared_error(test["helpful_vote"], pred)})

huber = HuberRegressor(max_iter=1000).fit(train[["log_words"]], train["helpful_vote"])
pred = huber.predict(test[["log_words"]])
rows.append({"model": "Huber", "slope": huber.coef_[0],
             "MAE": mean_absolute_error(test["helpful_vote"], pred),
             "RMSE": root_mean_squared_error(test["helpful_vote"], pred)})
pd.DataFrame(rows).set_index("model").round(3)

In [ ]:
grid = pd.DataFrame({"log_words": np.linspace(0, 7, 50)})
sub = train.sample(5000, random_state=1)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.scatter(sub["log_words"], sub["helpful_vote"], s=4, alpha=0.3, color="0.6", label="reviews")
for q, colour in [(0.5, "#009E73"), (0.9, "#0072B2")]:
    ax.plot(grid["log_words"], smf.quantreg("helpful_vote ~ log_words", train).fit(q=q).predict(grid),
            color=colour, lw=2, label=f"quantile {q}")
ax.plot(grid["log_words"], smf.ols("helpful_vote ~ log_words", train).fit().predict(grid),
        color="#D55E00", lw=2, ls="--", label="least squares (mean)")
ax.set(ylim=(-1, 25), xlabel="log(1 + words)", ylabel="helpful votes (axis cut at 25)",
       title="Mean, median and 90th percentile of votes by length")
ax.legend(frameon=False);

**Exercise.** (a) Repeat the comparison on the log scale (`log_votes`): do the methods still disagree? (b) Which model would you use to tell a product manager how many votes a *typical* review gets, and which to describe reviews that get *many* votes?

## Block 3 · Logistic regression: Telco churn

The IBM Telco sample (7,043 customers) is downloaded from GitHub (about 1 MB).

In [ ]:
url = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
       "master/data/Telco-Customer-Churn.csv")
telco = pd.read_csv(url)
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")   # 11 blanks -> NaN
telco["churn"] = (telco["Churn"] == "Yes").astype(int)
print(telco.shape, "churn rate:", round(telco["churn"].mean(), 3))
telco.groupby("Contract")["churn"].agg(["mean", "size"]).round(3)

In [ ]:
tr, te = train_test_split(telco, test_size=0.25, stratify=telco["churn"], random_state=0)
formula = ("churn ~ tenure + MonthlyCharges + SeniorCitizen + C(Contract) + C(InternetService)"
           " + C(PaymentMethod) + C(PaperlessBilling)")
logit = smf.logit(formula, data=tr).fit(disp=0)
coef = pd.DataFrame({"coef (log-odds)": logit.params, "odds ratio": np.exp(logit.params),
                     "OR low": np.exp(logit.conf_int()[0]), "OR high": np.exp(logit.conf_int()[1]),
                     "p": logit.pvalues})
coef.round(3)

**Exercise.** Interpret three coefficients as odds ratios (for example two-year contract, fibre optic, tenure per 12 months: `np.exp(12 * coef)`). Then translate one of them into probabilities for two typical customers with the cell below.

In [ ]:
typical = pd.DataFrame({
    "tenure": [3, 3], "MonthlyCharges": [80, 80], "SeniorCitizen": [0, 0],
    "Contract": ["Month-to-month", "Two year"], "InternetService": ["Fiber optic", "Fiber optic"],
    "PaymentMethod": ["Electronic check", "Electronic check"], "PaperlessBilling": ["Yes", "Yes"],
})
typical.assign(p_churn=logit.predict(typical).round(3))

### The same model in scikit-learn

In [ ]:
numeric = ["tenure", "MonthlyCharges", "SeniorCitizen"]
categorical = ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling"]
pre = ColumnTransformer([("num", "passthrough", numeric),
                         ("cat", OneHotEncoder(drop="first"), categorical)])
clf = make_pipeline(pre, LogisticRegression(C=np.inf, max_iter=5000)).fit(tr, tr["churn"])
sk_coef = pd.Series(clf[-1].coef_[0], index=clf[0].get_feature_names_out())
sk_coef.round(3)

### A first look at predicted classes

In [ ]:
p = clf.predict_proba(te)[:, 1]
print("majority baseline accuracy:", round(1 - te["churn"].mean(), 3))
rows = []
for threshold in [0.5, 0.4, 0.3, 0.2]:
    pred = (p > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(te["churn"], pred).ravel()
    rows.append({"threshold": threshold, "accuracy": (pred == te["churn"]).mean(),
                 "TP (churners found)": tp, "FN (missed)": fn, "FP (false alarms)": fp, "TN": tn})
pd.DataFrame(rows).set_index("threshold").round(3)

**Exercise.** A retention call costs €10; a lost customer costs €200 of margin. Which threshold minimises the total cost on the test set (cost = 10 × (TP + FP) + 200 × FN)? Session 8 returns to this question with ROC curves.